# Session 6: What a language model does

**259.075 Data-integrated Algorithmic Design Processes II** · Session 6 of 7

You have been using these things daily for two years. The point of today is that afterwards you
can say what happens inside one, in words you built yourself.

The model we train is small enough to fit in this notebook and finishes in about ten seconds. It
learns the street names of Vienna, letter by letter, and then invents new ones. Everything that
makes a large language model different from it is a matter of scale and two additions, and both
get named at the end.

<img src="img/strassenschild.jpg" width="330">

*Photo: Christian Michelides,
[Wikimedia Commons](https://commons.wikimedia.org/wiki/File:Stra%C3%9Fenschild_Br%C3%BCder-Heindl-Gasse_(Wien).jpg),
CC BY-SA 4.0.*

In [ ]:
import time
import numpy as np
import torch
import torch.nn as nn
import plotly.graph_objects as go
from plotly.subplots import make_subplots

namen = [z.strip() for z in
         open("../data/strassennamen.txt", encoding="utf-8").read().splitlines() if z.strip()]
print(f"{len(namen)} street names, {np.mean([len(n) for n in namen]):.1f} characters on average")
print(", ".join(namen[2000:2006]))

## Cutting text into pieces

A model computes with numbers, so the text has to be cut into pieces and each piece given a
number. The pieces are called *tokens*, and how you cut is a decision with consequences.

Cut into characters and the vocabulary is tiny, maybe eighty entries, but "Lerchenfelder" takes
thirteen steps to produce and the model has to learn from scratch that "gasse" is a thing. Cut
into whole words and every name is one or two steps, but the vocabulary is as large as the
dictionary and any word not in it is simply unrepresentable. Real systems cut into word
fragments, which is the compromise: common words stay whole, rare ones fall apart.

> A token is not a unit of meaning. It is a unit of accounting, chosen for convenience.

This also explains a failure you have probably seen. If a model cuts "1024" into "10" and "24",
it is not looking at a number at all, and arithmetic goes about as well as you would expect.

<img src="img/setzkasten.jpg" width="380">

*A typesetter's case: the vocabulary of an older generative system. Photo: VIGNERON,
[Wikimedia Commons](https://commons.wikimedia.org/wiki/File:Casse_de_typographe,_Grandjean_-_mus%C3%A9e_Champollion.JPG),
CC BY 3.0.*

In [ ]:
satz = "Lerchenfelder Gürtel"
stuecke = ["Lerchen", "felder", " Gür", "tel"]                   # what a real tokeniser does
for name, teile in (("characters", list(satz)),
                    ("words", satz.split()),
                    ("fragments", stuecke)):
    print(f"{name:11s} {len(teile):3d} tokens  {' | '.join(teile)}")

alle_worte = {w for n in namen for w in n.replace("-", " ").split()}
print(f"\nvocabulary if we cut into characters: "
      f"{len(set(''.join(namen)))}")
print(f"vocabulary if we cut into whole words : {len(alle_worte)}")

Eighty-two against seven thousand. For seven thousand names and ten seconds of training, characters
win, and everything below is about characters.

## A different question, and a different loss

Every model so far answered with a number, and the loss was the distance to the right number.
A model that writes cannot do that: there is no sense in which "g" is 0.3 away from "h".

So it answers with a *probability for every possible next token*. After "Gumpendorfer Stra" it
should put almost all of its confidence on "ß" and very little anywhere else. The loss asks how
surprised the model was by what actually came next, and the standard way to measure surprise is
the negative logarithm of the probability it assigned. That is *cross entropy*.

Three predictions, the same correct answer, three very different losses:

In [ ]:
moeglich = ["ß", "s", "n", "e"]
faelle = {"confident and right": [0.90, 0.04, 0.03, 0.03],
          "unsure": [0.30, 0.30, 0.20, 0.20],
          "confident and wrong": [0.02, 0.05, 0.90, 0.03]}

fig = make_subplots(rows=1, cols=3, subplot_titles=[
    f"{name}<br>loss {-np.log(p[0]):.2f}" for name, p in faelle.items()])
for spalte, (name, p) in enumerate(faelle.items(), start=1):
    farben = ["crimson" if z == "ß" else "#4c78a8" for z in moeglich]
    fig.add_trace(go.Bar(x=moeglich, y=p, marker_color=farben, showlegend=False),
                  row=1, col=spalte)
fig.update_layout(height=330, title="the true next character is ß")
fig.update_yaxes(range=[0, 1])
fig.show()

Being unsure is punished mildly. Being confident and wrong is punished savagely, and there is no
upper limit to it: assign zero probability to what actually happens and the loss is infinite.
That asymmetry is the whole reason these models hedge.

> Cross entropy does not ask whether the model was right. It asks how much probability it put on
> what happened.

## The model

Take the last eight characters, look each of them up in a small table of learned vectors, throw
the lot into the network from session 2, and get one score per character in the vocabulary.

That lookup table is worth a name: an *embedding*. Instead of feeding the model the number 43
for "m", which would suggest that "m" is somehow between 42 and 44, each character gets a short
vector of its own that the training adjusts. Characters used in similar places drift together.
The same idea, at a larger scale, is what makes "king" and "queen" end up near each other in a
language model.

In [ ]:
FENSTER = 8
ANFANG, ENDE = "^", "$"

zeichen = sorted(set("".join(namen)) | {ANFANG, ENDE})
zu_zahl = {z: i for i, z in enumerate(zeichen)}
V = len(zeichen)
print(f"{V} characters in the vocabulary")


def beispiele(liste):
    """Every window of eight characters, and the character that follows it."""
    X, Y = [], []
    for name in liste:
        folge = [zu_zahl[ANFANG]] * FENSTER + [zu_zahl[c] for c in name] + [zu_zahl[ENDE]]
        for i in range(FENSTER, len(folge)):
            X.append(folge[i - FENSTER:i])
            Y.append(folge[i])
    return torch.tensor(X), torch.tensor(Y)


zufall = np.random.default_rng(5)
i = np.arange(len(namen))
zufall.shuffle(i)
namen_tr = [namen[j] for j in i[:6000]]
namen_va = [namen[j] for j in i[6000:]]
Xtr, Ytr = beispiele(namen_tr)
Xva, Yva = beispiele(namen_va)
print(f"{len(Xtr)} training windows, {len(Xva)} validation windows")
print("one window:", "".join(zeichen[n] for n in Xtr[300].tolist()),
      "->", zeichen[Ytr[300]])

In [ ]:
class Zeichenmodell(nn.Module):
    def __init__(self, breite=16, verborgen=256):
        super().__init__()
        self.tabelle = nn.Embedding(V, breite)          # one short vector per character
        self.netz = nn.Sequential(nn.Linear(FENSTER * breite, verborgen), nn.ReLU(),
                                  nn.Linear(verborgen, V))

    def forward(self, x):
        return self.netz(self.tabelle(x).flatten(1))


torch.manual_seed(0)
modell = Zeichenmodell()
verlust = nn.CrossEntropyLoss()
optimierer = torch.optim.Adam(modell.parameters(), lr=2e-3)
print(f"{sum(p.numel() for p in modell.parameters()):,} parameters")

## Drawing a character

Training comes in a moment; first the part you write, because you need it to watch the training.

The model hands back one score per character, and those scores have to become a choice. Divide
by the temperature, turn into probabilities, draw one. Taking the highest-scoring character
every time would be a legitimate choice too, and it produces the same name forever, which is why
nobody does it.

In [ ]:
def ziehe_zeichen(punkte, temperatur=1.0, wuerfel=None):
    """One index, drawn from the scores. Higher temperature means a flatter distribution."""
    wuerfel = wuerfel or np.random.default_rng()

    # YOUR TURN: three lines. Divide the scores by the temperature, turn them into
    # probabilities (subtract the largest first, or the exponential overflows), and draw one
    # index from them with wuerfel.choice.
    raise NotImplementedError("sampling missing")



def ziehe_namen(modell, temperatur=1.0, wuerfel=None, laenge=40):
    wuerfel = wuerfel or np.random.default_rng()
    folge = [zu_zahl[ANFANG]] * FENSTER
    for _ in range(laenge):
        with torch.no_grad():
            punkte = modell(torch.tensor([folge[-FENSTER:]]))[0].numpy()
        n = ziehe_zeichen(punkte, temperatur, wuerfel)
        if zeichen[n] == ENDE:
            break
        folge.append(n)
    return "".join(zeichen[n] for n in folge[FENSTER:])


print("before training:", ziehe_namen(modell, wuerfel=np.random.default_rng(0)))

## Ten seconds

Train it. Every five hundred steps the cell prints the two losses and four names, so you can
watch the thing learn to spell.

In [ ]:
SCHRITTE = 3000
stellen, kurve_tr, kurve_va, zustaende = [], [], [], []
t0 = time.perf_counter()
for schritt in range(SCHRITTE):
    wahl = torch.randint(0, len(Xtr), (512,))
    fehler = verlust(modell(Xtr[wahl]), Ytr[wahl])
    optimierer.zero_grad()
    fehler.backward()
    optimierer.step()

    getan = schritt + 1
    if getan % 100 == 0 or getan == SCHRITTE:
        with torch.no_grad():
            f_va = float(verlust(modell(Xva), Yva))
        stellen.append(getan)
        kurve_tr.append(float(fehler))
        kurve_va.append(f_va)
        zustaende.append({n: t.detach().clone() for n, t in modell.state_dict().items()})
    if getan % 500 == 0:
        proben = [ziehe_namen(modell, wuerfel=np.random.default_rng(s)) for s in range(4)]
        print(f"{getan:5d}  train {float(fehler):.3f}  val {f_va:.3f}  "
              f"{time.perf_counter() - t0:4.0f}s  " + " · ".join(proben))

tief = int(np.argmin(kurve_va))
print(f"\nlowest validation loss {kurve_va[tief]:.3f} after {stellen[tief]} updates")
modell.load_state_dict(zustaende[tief])           # early stopping, as in session 3

In [ ]:
fig = go.Figure()
fig.add_trace(go.Scatter(x=stellen, y=kurve_tr, mode="lines", name="training",
                         line=dict(width=2, color="#4c78a8")))
fig.add_trace(go.Scatter(x=stellen, y=kurve_va, mode="lines", name="validation",
                         line=dict(width=3, color="crimson")))
fig.add_vline(x=stellen[tief], line_dash="dash", line_color="#d8a24a")
fig.update_layout(height=400, xaxis_title="updates", yaxis_title="cross entropy",
                  title="the same picture as session 3, on letters")
fig.show()

## What the model expects

Type something into the cell below and look at what it thinks comes next. This is the entire
mechanism of every text tool you use, drawn as a bar chart.

In [ ]:
def erwartung(bisher, oben=12):
    fremd = sorted(set(bisher) - set(zeichen))
    assert not fremd, f"not in the vocabulary of the street names: {' '.join(fremd)}"
    folge = ([zu_zahl[ANFANG]] * FENSTER + [zu_zahl[c] for c in bisher])[-FENSTER:]
    with torch.no_grad():
        punkte = modell(torch.tensor([folge]))[0].numpy()
    p = np.exp(punkte - punkte.max())
    p /= p.sum()
    reihe = np.argsort(p)[::-1][:oben]
    beschriftung = ["<end>" if zeichen[n] == ENDE else zeichen[n] for n in reihe]
    fig = go.Figure(go.Bar(x=beschriftung, y=p[reihe], marker_color="#4c78a8"))
    fig.update_layout(height=340, yaxis_title="probability",
                      title=f"after \"{bisher}\" the model expects")
    fig.show()


erwartung("Gumpendorfer Stra")

In [ ]:
erwartung("Siebenbrunnenga")

## The dial

Same model, six temperatures, twelve names each. Cold, it plays safe: the commonest letter
patterns, and a marked reluctance to produce the end-of-name character at all, so the names run
long and compound. Hot, it forgets how to spell. The band in between is narrow and you find it by
looking, which is exactly the situation you are in with every image and text tool on the market.

In [ ]:
temperaturen = [0.3, 0.6, 0.8, 1.0, 1.3, 1.8]
bekannt = set(namen)
gezogen = {}
for t in temperaturen:
    gezogen[t] = [ziehe_namen(modell, t, np.random.default_rng(1000 + s)) for s in range(12)]

for t in temperaturen:
    neu = [g for g in gezogen[t] if g not in bekannt]
    print(f"T = {t}: {len(neu):2d} of 12 are not real streets")
    print(f"         " + " · ".join(gezogen[t][:6]))

## Hallucination is not a bug

Count the last column again: at any useful temperature, almost every name the model produces is
not a street in Vienna. That is not a defect, it is the job. We asked for plausible continuations
and we got plausible continuations.

A large language model does exactly this, with word fragments instead of characters and a much
longer window. When it tells you that DIN 18040-2 requires a turning circle of 1.50 metres, it
is producing a plausible continuation of your question. Sometimes the plausible continuation is
also true, because true things appeared often in the training text. Sometimes it is not, and
nothing in the output distinguishes the two cases, because the mechanism is the same.

> A language model estimates the likely continuation. Likely and true are different properties,
> and only one of them is being optimised.

This is why the practical rule is neither "never use it" nor "it will get better". The rule is:
anything you would have to check if a stranger told it to you, you have to check here too, and
the confident tone carries no information at all.

## From here to the tools you use

Three differences, and only one of them is an idea rather than a quantity.

The quantity: word fragments instead of characters, billions of parameters instead of forty
thousand, and a window of a hundred thousand tokens instead of eight. Everything else about the
loop, the loss and the sampling is what you just ran.

The idea: *attention*. Our model looks at exactly the last eight characters, with one learned
weight per position, the same for every text.
Attention lets the model decide, for each position, which earlier positions matter, so a pronoun
can reach back to the noun it refers to across a paragraph. That is the mechanism the letter T
in GPT stands for, and it is the reason the window could grow at all.

The third: what turns a continuation machine into something that answers questions is training
afterwards on human judgements of good and bad answers. The base model completes text. The
assistant you talk to has been shaped, after the fact, to complete it in the form of a helpful
reply.

## Tests

In [ ]:
punkte = np.array([2.0, 1.0, 0.0, -1.0])

# cold draws must land on the highest score
w = np.random.default_rng(0)
kalt = [ziehe_zeichen(punkte, 0.01, w) for _ in range(30)]
assert set(kalt) == {0}, f"at temperature 0.01 only index 0 should come up, got {set(kalt)}"

# hot draws must spread out
w = np.random.default_rng(0)
heiss = [ziehe_zeichen(punkte, 5.0, w) for _ in range(200)]
assert len(set(heiss)) == 4, f"at temperature 5 all four should appear, got {set(heiss)}"

# the frequencies at temperature 1 must follow softmax
w = np.random.default_rng(1)
zaehler = np.bincount([ziehe_zeichen(punkte, 1.0, w) for _ in range(20000)], minlength=4) / 20000
soll = np.exp(punkte) / np.exp(punkte).sum()
assert np.allclose(zaehler, soll, atol=0.02), f"frequencies {zaehler.round(3)} against {soll.round(3)}"

# no overflow on large scores
assert ziehe_zeichen(np.array([900.0, 0.0]), 1.0, np.random.default_rng(0)) == 0, (
    "subtract the largest score before exponentiating")

# the trained model has to be better than guessing
with torch.no_grad():
    erreicht = float(verlust(modell(Xva), Yva))
assert erreicht < np.log(V) - 1.0, f"barely better than guessing: {erreicht:.2f} against {np.log(V):.2f}"
print(f"all green. cross entropy {erreicht:.3f}, blind guessing would be {np.log(V):.3f}")

## Where you stand now

You can explain what a token is and why the choice of tokeniser leaks into what a model can do.
You know why a model that writes needs a different loss, and what cross entropy punishes. You
have trained a language model, watched it learn to spell, and turned its temperature. And you can
say, without hand-waving, why these systems produce confident falsehoods: because confidence and
truth are not the same axis, and only one of them is in the loss.

Next session: images, the models that draw them, and the trick that lets a small studio adapt a
large model on a laptop.

---

### Terms to learn

| term | in one sentence |
|---|---|
| token | the piece a text is cut into before numbering; a unit of accounting, not of meaning |
| vocabulary | the set of tokens a model can produce at all |
| embedding | a short learned vector standing for a token, so that similar tokens end up near each other |
| cross entropy | the loss for predicting a distribution; it punishes confident mistakes hardest |
| autoregressive | producing one token at a time, each conditioned on what came before |
| context window | how far back the model is allowed to look |
| attention | letting the model choose which earlier positions matter, instead of a fixed window |
| temperature | how far down the ranking the sampling is willing to go |
| hallucination | a plausible continuation that happens not to be true; the same mechanism as a correct one |
| fine-tuning on human feedback | what turns a continuation machine into an assistant |

## Homework

Ten points, due before the next session: four for running it, six for your thoughts on it.

**How to do it**

1. Fill in the `YOUR TURN` gap above, yourself or from the solution notebook. If you took it from
   the solution, say so in one line at the top of your answers; that costs nothing.
2. Below, change only the lines under `# YOUR CHOICE`.
3. Write your answers into the last cell.
4. Choose Kernel, Restart Kernel and Run All Cells, check that everything runs, and upload the
   notebook in TUWEL.

**How it is graded**

- Four points for the run: it runs without an error (one), it shows what the task asks for (one),
  and every number you give about your run appears in your output or is worked out from it in a
  step you write down (two). Numbers the questions give you, examples from outside the notebook
  and forecasts you mark as your own expectation are exempt.
- Six points for your answers, one per question, no halves. Say what you saw and what it means,
  and you get the point, also with borrowed code. A sentence that would fit any model gets none.

### The task

1. Train the same character model on a vocabulary of your own: plant names, architects,
   materials, place names from where you grew up, terms from your thesis. Paste it into the
   first cell, one entry per line, at least 200 distinct entries. A list page on Wikipedia, a
   catalogue or the index of a book will do. The smaller the list, the more the model tends to
   reproduce it instead of inventing.
2. Choose three temperatures, draw twenty words at each and show them.
3. Before you look at the table, write down in one sentence what you would use such invented
   words for, and what would make one usable for that. Then judge the table by that sentence.
4. Give one case from your own studies in which a language model told you something plausible
   and wrong.

In [ ]:
# YOUR CHOICE: paste your vocabulary between the two lines of quotes, one entry per line
liste = '''
'''

eigene = sorted({z.strip() for z in liste.splitlines() if z.strip()})
assert len(eigene) >= 200, f"{len(eigene)} distinct entries; at least 200 are needed"
assert not any(ANFANG in w or ENDE in w for w in eigene), "the characters ^ and $ are reserved"

zeichen_h = sorted(set("".join(eigene)) | {ANFANG, ENDE})
zu_zahl_h = {z: i for i, z in enumerate(zeichen_h)}


def fenster_h(liste):
    X, Y = [], []
    for wort in liste:
        folge = [zu_zahl_h[ANFANG]] * FENSTER + [zu_zahl_h[c] for c in wort] + [zu_zahl_h[ENDE]]
        for i in range(FENSTER, len(folge)):
            X.append(folge[i - FENSTER:i])
            Y.append(folge[i])
    return torch.tensor(X), torch.tensor(Y)


class Wortmodell(nn.Module):
    def __init__(self, v, breite=16, verborgen=256):
        super().__init__()
        self.tabelle = nn.Embedding(v, breite)
        self.netz = nn.Sequential(nn.Linear(FENSTER * breite, verborgen), nn.ReLU(),
                                  nn.Linear(verborgen, v))

    def forward(self, x):
        return self.netz(self.tabelle(x).flatten(1))


mischung = np.random.default_rng(5).permutation(len(eigene))
grenze = int(0.85 * len(eigene))
woerter_tr = [eigene[j] for j in mischung[:grenze]]
woerter_va = [eigene[j] for j in mischung[grenze:]]
Xh_tr, Yh_tr = fenster_h(woerter_tr)
Xh_va, Yh_va = fenster_h(woerter_va)

torch.manual_seed(0)
eigenes = Wortmodell(len(zeichen_h))
opt_h = torch.optim.Adam(eigenes.parameters(), lr=2e-3)
stellen_h, kurve_h, zust_h = [], [], []
for schritt in range(3000):
    wahl = torch.randint(0, len(Xh_tr), (512,))
    f = verlust(eigenes(Xh_tr[wahl]), Yh_tr[wahl])
    opt_h.zero_grad()
    f.backward()
    opt_h.step()
    if (schritt + 1) % 100 == 0:
        with torch.no_grad():
            kurve_h.append(float(verlust(eigenes(Xh_va), Yh_va)))
        stellen_h.append(schritt + 1)
        zust_h.append({n: t.detach().clone() for n, t in eigenes.state_dict().items()})
tief_h = int(np.argmin(kurve_h))
eigenes.load_state_dict(zust_h[tief_h])
print(f"{len(eigene)} entries, {len(zeichen_h)} characters, "
      f"{sum(p.numel() for p in eigenes.parameters()):,} parameters")
print(f"lowest validation loss {kurve_h[tief_h]:.3f} after {stellen_h[tief_h]} updates, "
      f"blind guessing {np.log(len(zeichen_h)):.3f}")
print(f"perplexity: model {np.exp(kurve_h[tief_h]):.1f}, blind guessing {len(zeichen_h)}")

In [ ]:
laenge_h = max(len(w) for w in eigene) + 10


def ziehe_wort(modell, temperatur, wuerfel):
    folge = [zu_zahl_h[ANFANG]] * FENSTER
    for _ in range(laenge_h):
        with torch.no_grad():
            punkte = modell(torch.tensor([folge[-FENSTER:]]))[0].numpy()
        punkte[zu_zahl_h[ANFANG]] = -np.inf             # the start marker is never a letter
        n = ziehe_zeichen(punkte, temperatur, wuerfel)
        if zeichen_h[n] == ENDE:
            return "".join(zeichen_h[n] for n in folge[FENSTER:]), False
        folge.append(n)
    return "".join(zeichen_h[n] for n in folge[FENSTER:]), True


# YOUR CHOICE: three different temperatures, each between 0.05 and 5
temperaturen_h = [0.5, 1.0, 1.5]

assert len(temperaturen_h) == 3 and len(set(temperaturen_h)) == 3, "three different temperatures"
assert all(0.05 <= t <= 5 for t in temperaturen_h), "each between 0.05 and 5"
bekannt_h = set(eigene)
spalten = {t: [ziehe_wort(eigenes, t, np.random.default_rng(100 + s)) for s in range(20)]
           for t in temperaturen_h}


def zeile(wort, abgeschnitten):
    if abgeschnitten:
        return wort + "  (cut off)"
    return wort + ("  (copy)" if wort in bekannt_h else "")


fig = go.Figure(go.Table(
    header=dict(values=[f"T = {t}: {sum(w in bekannt_h for w, _ in spalten[t])} copies, "
                        f"{sum(a for _, a in spalten[t])} cut off" for t in temperaturen_h]),
    cells=dict(values=[[zeile(w, a) for w, a in spalten[t]] for t in temperaturen_h],
               height=24)))
fig.update_layout(height=640, title="twenty draws at each temperature; copies of your list and "
                                    "words that never ended are marked")
fig.show()

### Your interpretation

1. **Size.** Give three printed numbers: your lowest validation loss, and the perplexity of the model and of blind guessing. Perplexity is e to the power of the loss: the number of equally likely characters that would leave the model just as uncertain. Compare the two perplexities and say what the difference means.
2. **Plausibility.** Do the outputs look like your vocabulary? Give two observations, each either a feature of your list the model picked up or one it visibly failed to, each with an example word from the table.
3. **Units.** The loss is averaged per character, not per word. Say what that makes comparable, and what an average over characters hides.
4. **Range.** Your sentence on what the words are for, then: at which of your temperatures do the outputs stop being usable for that, judged from the table? If all three are usable, or none, say so.
5. **Copies.** How many of your draws are copies of your list, at each temperature? Say what that suggests about your list size and the number of parameters, and why the table alone cannot prove that the model memorised.
6. **Consequence for your work.** One case from your own studies where a language model told you something plausible and wrong: what it said, why it sounded right, and how you could have noticed.

*Your text here.*